## Closed-Loop Prediction Mapping

This notebook compares the actual recorded end-effector trajectory with the closed-loop predicted motion path from `offline_prediction.py`.


In [1]:
# Load actual recorded trajectory and closed-loop prediction rollout.
# This keeps the import/loading style self-contained so the notebook can run by itself.

from pathlib import Path

import numpy as np
from scipy.spatial.transform import Rotation as R

try:
    import vedo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vedo is not installed in this environment. "
        "Activate blackwell-train, then run: pip install vedo"
    ) from exc

actual_npz_candidates = [
    Path("./results/grasp_cube/offline_predictions.npz"),
    Path("test_inference/results/grasp_cube/offline_predictions.npz"),
]
closed_loop_npz_candidates = [
    Path("./results/grasp_cube_predictions/closed_loop_predictions.npz"),
    Path("test_inference/results/grasp_cube_predictions/closed_loop_predictions.npz"),
]

actual_npz_path = next((path for path in actual_npz_candidates if path.exists()), None)
closed_loop_npz_path = next((path for path in closed_loop_npz_candidates if path.exists()), None)

if actual_npz_path is None:
    raise FileNotFoundError(
        "Could not find offline_predictions.npz. Tried: "
        + ", ".join(str(path) for path in actual_npz_candidates)
    )
if closed_loop_npz_path is None:
    raise FileNotFoundError(
        "Could not find closed_loop_predictions.npz. Tried: "
        + ", ".join(str(path) for path in closed_loop_npz_candidates)
    )

actual_data = np.load(actual_npz_path)
closed_loop_data = np.load(closed_loop_npz_path)

print("actual file:", actual_npz_path)
print("closed-loop file:", closed_loop_npz_path)
print("actual keys:", actual_data.files)
print("closed-loop keys:", closed_loop_data.files)


actual file: results/grasp_cube/offline_predictions.npz
closed-loop file: results/grasp_cube_predictions/closed_loop_predictions.npz
actual keys: ['pred_action', 'gt_action', 'dataset_index', 'episode_id', 'episode_timestep', 'before_first_grasp', 'base_eef_pos', 'base_eef_rot_axis_angle']
closed-loop keys: ['episode_id', 'input_timestep', 'matched_timestep', 'used_horizon_index', 'raw_action_horizon', 'abs_action_horizon', 'pred_eef_pos', 'pred_eef_rot_axis_angle', 'pred_gripper_width', 'gt_eef_pos', 'gt_eef_rot_axis_angle', 'gt_gripper_width']


In [ ]:
# Plot actual trajectory and predicted closed-loop motion path in 3D.
# Black path: actual recorded end-effector trajectory.
# Blue path: predicted motion path, built from every point in each predicted horizon.
# Red/green/blue arrows: local x/y/z orientation axes.

closed_loop_episode_id = closed_loop_data["episode_id"]
demo_id = int(closed_loop_episode_id[0])

# Actual recorded path for the same demo.
actual_episode_id = actual_data["episode_id"]
actual_episode_timestep = actual_data["episode_timestep"]
actual_mask = actual_episode_id == demo_id
if not np.any(actual_mask):
    raise ValueError(f"No actual recorded trajectory found for episode {demo_id}.")

actual_order = np.argsort(actual_episode_timestep[actual_mask])
actual_indices = np.where(actual_mask)[0][actual_order]
actual_xyz = actual_data["base_eef_pos"][actual_indices]
actual_rotvec = actual_data["base_eef_rot_axis_angle"][actual_indices]

# Predicted motion path from the closed-loop rollout.
# Important: this uses the full abs_action_horizon, not just pred_eef_pos.
# That means the blue line shows the predicted motion inside each rollout segment.
closed_mask = closed_loop_episode_id == demo_id
closed_order = np.argsort(closed_loop_data["input_timestep"][closed_mask])
closed_indices = np.where(closed_mask)[0][closed_order]

# These are the exact policy-call roots. At each one, offline_prediction.py
# reads real RGB frames from the dataset, then predicts the next horizon.
input_timesteps = closed_loop_data["input_timestep"][closed_indices]
if "dataset_index" in actual_data.files:
    dataset_index_to_row = {
        int(dataset_index): row
        for row, dataset_index in enumerate(actual_data["dataset_index"])
    }
    visual_input_rows = [
        dataset_index_to_row[int(timestep)]
        for timestep in input_timesteps
        if int(timestep) in dataset_index_to_row
    ]
    visual_input_xyz = actual_data["base_eef_pos"][visual_input_rows]
else:
    # Fallback: input_timestep is global, episode_timestep is local.
    input_local_timesteps = input_timesteps - input_timesteps[0]
    actual_local_to_path_row = {
        int(timestep): row
        for row, timestep in enumerate(actual_episode_timestep[actual_indices])
    }
    visual_input_path_rows = [
        actual_local_to_path_row[int(timestep)]
        for timestep in input_local_timesteps
        if int(timestep) in actual_local_to_path_row
    ]
    visual_input_xyz = actual_xyz[visual_input_path_rows]

predicted_xyz_parts = [actual_xyz[0:1]]
predicted_rotvec_parts = [actual_rotvec[0:1]]
for idx in closed_indices:
    used_horizon_index = int(closed_loop_data["used_horizon_index"][idx])
    horizon = closed_loop_data["abs_action_horizon"][idx]
    predicted_xyz_parts.append(horizon[:used_horizon_index + 1, :3])
    predicted_rotvec_parts.append(horizon[:used_horizon_index + 1, 3:6])

predicted_xyz = np.concatenate(predicted_xyz_parts, axis=0)
predicted_rotvec = np.concatenate(predicted_rotvec_parts, axis=0)

num_markers = 25
actual_marker_positions = np.unique(
    np.linspace(0, len(actual_xyz) - 1, min(num_markers, len(actual_xyz)), dtype=int)
)
predicted_marker_positions = np.unique(
    np.linspace(0, len(predicted_xyz) - 1, min(num_markers, len(predicted_xyz)), dtype=int)
)

actual_marker_xyz = actual_xyz[actual_marker_positions]
actual_marker_rot = R.from_rotvec(actual_rotvec[actual_marker_positions]).as_matrix()
predicted_marker_xyz = predicted_xyz[predicted_marker_positions]
predicted_marker_rot = R.from_rotvec(predicted_rotvec[predicted_marker_positions]).as_matrix()

axis_scale = 0.025

def axis_ends(marker_xyz, marker_rot):
    x_ends = marker_xyz + marker_rot[:, :, 0] * axis_scale
    y_ends = marker_xyz + marker_rot[:, :, 1] * axis_scale
    z_ends = marker_xyz + marker_rot[:, :, 2] * axis_scale
    return x_ends, y_ends, z_ends

actual_x_ends, actual_y_ends, actual_z_ends = axis_ends(actual_marker_xyz, actual_marker_rot)
pred_x_ends, pred_y_ends, pred_z_ends = axis_ends(predicted_marker_xyz, predicted_marker_rot)

actors = [
    vedo.Line(actual_xyz, c="black", lw=5, alpha=0.9),
    vedo.Points(actual_marker_xyz, c="black", r=8, alpha=1.0),
    vedo.Arrows(actual_marker_xyz, actual_x_ends, c="red", alpha=0.45),
    vedo.Arrows(actual_marker_xyz, actual_y_ends, c="green", alpha=0.45),
    vedo.Arrows(actual_marker_xyz, actual_z_ends, c="blue", alpha=0.45),
    vedo.Line(predicted_xyz, c="#1f77b4", lw=5, alpha=1.0),
    vedo.Points(predicted_marker_xyz, c="#1f77b4", r=9, alpha=1.0),
    vedo.Points(visual_input_xyz, c="yellow", r=16, alpha=1.0),
    vedo.Arrows(predicted_marker_xyz, pred_x_ends, c="red", alpha=0.9),
    vedo.Arrows(predicted_marker_xyz, pred_y_ends, c="green", alpha=0.9),
    vedo.Arrows(predicted_marker_xyz, pred_z_ends, c="blue", alpha=0.9),
]

print("demo id:", demo_id)
print("actual trajectory points:", len(actual_xyz))
print("predicted trajectory points:", len(predicted_xyz))
print("closed-loop rollout steps:", len(closed_indices))
print("real visual input roots:", len(visual_input_xyz))
print("input timesteps:", input_timesteps.tolist())
print("black = actual recorded trajectory")
print("blue = predicted closed-loop motion path from abs_action_horizon")
print("yellow dots = real visual-data refresh points where the model was called")
print("faint arrows = actual orientation; bright arrows = predicted orientation")

vedo.settings.default_backend = "vtk"
vedo.show(
    actors,
    axes=1,
    viewup="z",
    bg="white",
    title=f"Actual vs Closed-Loop Predicted Motion: Demo {demo_id}",
)


demo id: 0
actual trajectory points: 245
predicted trajectory points: 105
closed-loop rollout steps: 7
real visual input roots: 7
input timesteps: [0, 45, 90, 135, 180, 225, 270]
black = actual recorded trajectory
blue = predicted closed-loop motion path from abs_action_horizon
yellow dots = real visual-data refresh points where the model was called
faint arrows = actual orientation; bright arrows = predicted orientation
